In [ ]:
!git clone https://github.com/Ish2905/Comemo-Dataset.git

In [ ]:
%cd /content/Comemo-Dataset

In [ ]:
!git pull

In [ ]:
# ==== DO NOT MODIFY THIS CELL ====
from google.colab import drive
drive.mount('/content/drive')

import duckdb
import os

DB_PATH = "/content/drive/MyDrive/Capstone/comemo.db"

# HARD FAIL if Drive is not mounted
assert os.path.exists("/content/drive/MyDrive"), "Drive not mounted!"

# HARD FAIL if DB file missing (after first creation)
if not os.path.exists(DB_PATH):
    print("⚠️ comemo.db not found yet (first run only)")
else:
    print("✅ Using existing database:", DB_PATH)

con = duckdb.connect(DB_PATH)

# Sanity check
print(con.execute("SHOW TABLES").fetchdf())
# =================================


In [ ]:
con.execute("""COPY (
    WITH base AS (
        SELECT
            parent_asin,
            rating,
            DATE_TRUNC(
                'month',
                to_timestamp(timestamp / 1000)
            ) AS month_start
        FROM reviews_raw
    ),
    first_review AS (
        SELECT
            parent_asin,
            MIN(month_start) AS first_month
        FROM base
        GROUP BY parent_asin
    )
    SELECT
        b.parent_asin,
        EXTRACT(year FROM b.month_start) AS year,
        EXTRACT(month FROM b.month_start) AS month,
        STRFTIME(b.month_start, '%Y-%m') AS year_month,
        COUNT(*) AS monthly_review_count,
        AVG(b.rating) AS monthly_rating_avg,
        DATE_DIFF('month', f.first_month, b.month_start) AS product_age_months
    FROM base b
    JOIN first_review f
        ON b.parent_asin = f.parent_asin
    GROUP BY
        b.parent_asin,
        year,
        month,
        year_month,
        b.month_start,
        f.first_month
)
TO '/content/drive/MyDrive/Capstone/reviews_monthly.parquet'
(FORMAT PARQUET);""")


In [ ]:
con.execute("""
SELECT *
FROM read_parquet('/content/drive/MyDrive/Capstone/reviews_monthly.parquet')
LIMIT 10;
""").fetchdf()


In [ ]:
con.execute("""COPY (
    SELECT
        *,
        (monthly_review_count -
         LAG(monthly_review_count) OVER (
            PARTITION BY parent_asin
            ORDER BY year_month
         )) /
        NULLIF(
            LAG(monthly_review_count) OVER (
                PARTITION BY parent_asin
                ORDER BY year_month
            ), 0
        ) AS review_growth_rate,

        monthly_review_count / 30.0 AS review_velocity,

        monthly_rating_avg -
        LAG(monthly_rating_avg) OVER (
            PARTITION BY parent_asin
            ORDER BY year_month
        ) AS monthly_rating_change

    FROM read_parquet(
        '/content/drive/MyDrive/Capstone/reviews_monthly.parquet'
    )
)
TO '/content/drive/MyDrive/Capstone/reviews_momentum.parquet'
(FORMAT PARQUET);
""")


In [ ]:
con.execute("""
SELECT *
FROM read_parquet('/content/drive/MyDrive/Capstone/reviews_momentum.parquet')
LIMIT 10;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    monthly_review_count,
    COUNT(*) AS freq
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_monthly.parquet'
)
GROUP BY monthly_review_count
ORDER BY monthly_review_count;
""").fetchdf()


In [ ]:
con.execute("""SELECT *
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_monthly.parquet'
)
WHERE monthly_review_count > 20
ORDER BY monthly_review_count DESC
LIMIT 10;""").fetchdf()


In [ ]:
con.execute("""COPY (
    WITH trust_base AS (
        SELECT
            parent_asin,
            STRFTIME(
                DATE_TRUNC('month', to_timestamp(timestamp / 1000)),
                '%Y-%m'
            ) AS year_month,
            AVG(CAST(verified_purchase AS INT)) AS verified_ratio,
            SUM(helpful_vote) * 1.0 / COUNT(*) AS engagement_score
        FROM reviews_raw
        GROUP BY parent_asin, year_month
    )
    SELECT
        m.*,
        t.verified_ratio,
        t.engagement_score
    FROM read_parquet(
        '/content/drive/MyDrive/Capstone/reviews_momentum.parquet'
    ) m
    LEFT JOIN trust_base t
        ON m.parent_asin = t.parent_asin
       AND m.year_month = t.year_month
)
TO '/content/drive/MyDrive/Capstone/reviews_trust.parquet'
(FORMAT PARQUET);""")


In [ ]:
con.execute("""
SELECT *
FROM read_parquet('/content/drive/MyDrive/Capstone/reviews_trust.parquet')
LIMIT 10;
""").fetchdf()

In [ ]:
con.execute("""SELECT *
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_trust.parquet'
)
WHERE monthly_review_count > 20
ORDER BY monthly_review_count DESC
LIMIT 10;""").fetchdf()

In [ ]:
con.execute("""COPY (
    WITH category_avg AS (
        SELECT
            m.main_category,
            AVG(r.review_growth_rate) AS category_avg_growth
        FROM read_parquet(
            '/content/drive/MyDrive/Capstone/reviews_trust.parquet'
        ) r
        JOIN metadata_raw m
            ON r.parent_asin = m.parent_asin
        GROUP BY m.main_category
    )
    SELECT
        r.*,
        m.main_category,
        c.category_avg_growth,
        r.review_growth_rate - c.category_avg_growth
            AS relative_trend_score
    FROM read_parquet(
        '/content/drive/MyDrive/Capstone/reviews_trust.parquet'
    ) r
    JOIN metadata_raw m
        ON r.parent_asin = m.parent_asin
    JOIN category_avg c
        ON m.main_category = c.main_category
)
TO '/content/drive/MyDrive/Capstone/reviews_category.parquet'
(FORMAT PARQUET);""")


In [ ]:
con.execute("""
SELECT
    main_category,
    AVG(relative_trend_score) AS avg_rel_trend
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_category.parquet'
)
GROUP BY main_category
ORDER BY avg_rel_trend DESC
LIMIT 10;
""").fetchdf()


In [ ]:
con.execute("""
SELECT
    main_category,
    COUNT(*) AS row_count
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_category.parquet'
)
GROUP BY main_category
ORDER BY row_count DESC;""").fetchdf()


In [ ]:
con.execute("""SELECT *
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_category.parquet'
)
WHERE monthly_review_count > 20
ORDER BY monthly_review_count DESC
LIMIT 10;""").fetchdf()

In [ ]:
con.execute("""COPY (
    SELECT
        *,
        CASE
            WHEN product_age_months <= 6
                 AND relative_trend_score > 0
            THEN 'emerging'

            WHEN relative_trend_score > 0.2
                 AND review_growth_rate > 0
            THEN 'growing'

            WHEN monthly_review_count >= 20
                 AND relative_trend_score BETWEEN -0.05 AND 0.2
            THEN 'peaking'

            WHEN ABS(relative_trend_score) < 0.05
            THEN 'stable'

            ELSE 'declining'
        END AS trend_lifecycle_stage
    FROM read_parquet(
        '/content/drive/MyDrive/Capstone/reviews_category.parquet'
    )
)
TO '/content/drive/MyDrive/Capstone/reviews_lifecycle.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT *
FROM read_parquet(
  '/content/drive/MyDrive/Capstone/reviews_lifecycle.parquet'
)
LIMIT 10;
""").fetchdf()


In [ ]:
con.execute("""SELECT
    trend_lifecycle_stage,
    COUNT(*) AS count
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_lifecycle.parquet'
)
GROUP BY trend_lifecycle_stage
ORDER BY count DESC;
""").fetchdf()

In [ ]:
con.execute("DESC metadata_raw").fetchdf()

In [ ]:
con.execute("DESC reviews_raw").fetchdf()

In [ ]:
con.execute("""
COPY reviews_raw
TO '/content/drive/MyDrive/Capstone/reviews_raw.parquet'
(FORMAT PARQUET);

COPY metadata_raw
TO '/content/drive/MyDrive/Capstone/metadata_raw.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("PRAGMA threads=4;")
con.execute("PRAGMA enable_progress_bar=true;")



In [ ]:
con.execute("""
DROP TABLE IF EXISTS reviews_monthly;
CREATE TABLE reviews_monthly AS
WITH base AS (
    SELECT
        parent_asin,
        DATE_TRUNC('month', to_timestamp(timestamp)) AS month_start,
        rating
    FROM reviews_raw
),

first_review AS (
    SELECT
        parent_asin,
        MIN(month_start) AS first_month
    FROM base
    GROUP BY parent_asin
)

SELECT
    b.parent_asin,

    EXTRACT(year FROM b.month_start) AS year,
    EXTRACT(month FROM b.month_start) AS month,
    STRFTIME(b.month_start, '%Y-%m') AS year_month,

    COUNT(*) AS monthly_review_count,
    AVG(b.rating) AS monthly_rating_avg,

    DATE_DIFF(
        'month',
        f.first_month,
        b.month_start
    ) AS product_age_months

FROM base b
JOIN first_review f
    ON b.parent_asin = f.parent_asin

GROUP BY
    b.parent_asin,
    year,
    month,
    year_month,
    b.month_start,
    f.first_month;
""")

In [ ]:
con.execute("SHOW TABLES").fetchdf()


In [ ]:
con.execute("""CREATE TABLE reviews_monthly AS
SELECT * FROM read_parquet('/content/drive/MyDrive/Capstone/reviews_monthly.parquet');""")


In [ ]:
con.execute("DESC reviews_monthly").fetchdf()

In [ ]:
con.close()

In [ ]:
con.execute("""
COPY reviews_monthly
TO '/content/drive/MyDrive/Capstone/reviews_monthly.parquet'
(FORMAT PARQUET);
""")


In [ ]:
import duckdb

DB_PATH = "/content/drive/MyDrive/Capstone/comemo.db"
META_PATH = "/content/drive/MyDrive/Capstone/comemo_data/metadata.jsonl"

con = duckdb.connect(DB_PATH)

print("♻️ Creating CLEAN metadata_raw...")

con.execute(f"""
CREATE OR REPLACE TABLE metadata_raw AS
SELECT *
FROM read_json(
    '{META_PATH}',
    format='newline_delimited',
    ignore_errors=true,
    sample_size=-1
)
""")

print("✅ metadata_raw created successfully!")

# Verify columns
print(con.execute("DESCRIBE metadata_raw").fetchdf())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.listdir('/content/drive')

In [ ]:
os.listdir('/content/drive/MyDrive')

In [ ]:
!pip install open_clip_torch chromadb duckdb

import os, shutil, requests, datetime
from concurrent.futures import ThreadPoolExecutor
import chromadb
import torch
import open_clip
from PIL import Image
import duckdb

In [ ]:
DB_PATH = "/content/drive/MyDrive/Capstone/comemo.db"
CHROMA_PATH = "/content/drive/MyDrive/Capstone/chroma_db"
IMAGE_FOLDER = "/content/images"

BATCH_SIZE_DB = 2000
EMBED_BATCH_SIZE = 64

# Reset DB (optional)
shutil.rmtree(CHROMA_PATH, ignore_errors=True)

os.makedirs(IMAGE_FOLDER, exist_ok=True)

client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection("products")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("🚀 Using device:", device)

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)

model.to(device)
model.eval()

print("✅ CLIP model loaded")

In [ ]:
start = int(datetime.datetime(2022,1,1).timestamp() * 1000)
end   = int(datetime.datetime(2024,1,1).timestamp() * 1000)

print("✅ Time filter ready")

In [ ]:
import duckdb

DB_PATH = "/content/drive/MyDrive/Capstone (1)/comemo.db"

con = duckdb.connect(DB_PATH)

print("✅ DuckDB connected")

In [ ]:
import os

print("Capstone:", os.listdir('/content/drive/MyDrive/Capstone'))
print("Capstone (1):", os.listdir('/content/drive/MyDrive/Capstone (1)'))

In [ ]:
os.listdir('/content/drive/MyDrive/Capstone (1)/comemo_data')

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE metadata_raw AS
SELECT *
FROM read_json(
    '/content/drive/MyDrive/Capstone (1)/comemo_data/metadata.jsonl',
    format='newline_delimited',
    ignore_errors=true,
    sample_size=-1
)
""")

In [ ]:
query = f"""
SELECT m.parent_asin, m.images
FROM metadata_raw m
JOIN (
    SELECT parent_asin
    FROM reviews_raw
    WHERE timestamp >= {start}
      AND timestamp < {end}
    GROUP BY parent_asin
    HAVING COUNT(*) >= 10   -- 🔥 stricter filter
               -- 🔥 smaller but stronger dataset
) r
ON m.parent_asin = r.parent_asin
WHERE m.images IS NOT NULL
"""

In [ ]:
count = con.execute(f"""
SELECT COUNT(*)
FROM (
    {query}
)
""").fetchone()[0]

print("Filtered products:", count)

In [ ]:
import duckdb
import os, shutil, requests
from concurrent.futures import ThreadPoolExecutor
import chromadb
import torch
import open_clip
from PIL import Image
import datetime



# Re-defining start and end to ensure they are available in this cell
start = int(datetime.datetime(2022,1,1).timestamp() * 1000)
end   = int(datetime.datetime(2024,1,1).timestamp() * 1000)

query = f"""
SELECT m.parent_asin, m.images
FROM metadata_raw m
JOIN (
    SELECT parent_asin
    FROM reviews_raw
    WHERE timestamp >= {start}
      AND timestamp < {end}
    GROUP BY parent_asin
    HAVING COUNT(*) >= 10   -- 🔥 stricter filter
               -- 🔥 smaller but stronger dataset
) r
ON m.parent_asin = r.parent_asin
WHERE m.images IS NOT NULL
"""

def download_image(args):
    pid, url = args
    try:
        response = requests.get(url, stream=True, timeout=5)
        response.raise_for_status()
        with open(os.path.join(IMAGE_FOLDER, f"{pid}_{os.path.basename(url.split('?')[0])}"), 'wb') as out_file:
            shutil.copyfileobj(response.raw, out_file)
        del response
        return True
    except Exception:
        return False

def download_parallel(image_data):
    with ThreadPoolExecutor(max_workers=10) as executor:
        executor.map(download_image, image_data)

def get_embedding_batch(paths):
    images = []
    valid_paths = []
    for path in paths:
        try:
            img = Image.open(path).convert("RGB")
            images.append(preprocess(img))
            valid_paths.append(path)
        except Exception:
            # print(f"Warning: Could not load image {path}")
            continue

    if not images:
        return [], []

    with torch.no_grad():
        image_input = torch.tensor(torch.stack(images)).to(device)
        image_features = model.encode_image(image_input)
        image_features /= image_features.norm(dim=-1, keep_leaf=True)

    return image_features.cpu(), valid_paths

total = 0

cursor = con.execute(query) # Initialize cursor here

while True:

    batch = cursor.fetchmany(2000)   # stream from DuckDB

    if not batch:
        print("✅ Finished all data")
        break

    print(f"\n🚀 Processing batch size: {len(batch)}")

    image_data = []

    # -------- Extract images --------
    for row in batch:
        pid = row[0]
        imgs = row[1]

        if not imgs:
            continue

        try:
            # handle struct/list
            img = imgs if not isinstance(imgs, list) else imgs[0]

            url = None
            if isinstance(img, dict):
                url = img.get("hi_res") or img.get("large")

            if url:
                image_data.append((pid, url))

        except:
            continue

    if len(image_data) == 0:
        continue

    # -------- Download images --------
    download_parallel(image_data)

    # -------- Embed + Store --------
    files = os.listdir(IMAGE_FOLDER)
    success = 0

    for i in range(0, len(files), EMBED_BATCH_SIZE):

        batch_files = files[i:i+EMBED_BATCH_SIZE]
        paths = [os.path.join(IMAGE_FOLDER, f) for f in batch_files]

        embeddings, valid_paths = get_embedding_batch(paths)

        if len(embeddings) == 0:
            continue

        ids = []
        metas = []

        for j, emb in enumerate(embeddings):
            file = os.path.basename(valid_paths[j])
            pid = file.split("_")[0]

            ids.append(f"{file}_{total}_{j}")
            metas.append({"product_id": pid})

        try:
            collection.add(
                embeddings=[e.tolist() for e in embeddings],
                ids=ids,
                metadatas=metas
            )
            success += len(ids)

        except:
            continue

    total += success

    print(f"✅ Batch stored: {success}")
    print(f"📦 Total embeddings: {collection.count()}")

    # -------- Cleanup --------
    shutil.rmtree(IMAGE_FOLDER)
    os.makedirs(IMAGE_FOLDER, exist_ok=True)